In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "analysis":
    os.chdir(os.path.dirname(os.getcwd()))
    sys.path.append(os.getcwd())


In [7]:
from pathlib import Path
import re
import pandas as pd

experiment_results_dirs = [
    (
        "../logs/full_runs/2026_03_10_kodak_rgb/results",
        "2026_03_10_kodak_rgb",
    ),
    (
        "../logs_cluster/logs/full_runs/december/03_12_2025_rgb_arm_no_color_regression_kodak/results",
        "03_12_2025_rgb_arm_no_color_regression_kodak",
    ),
]

required_text_sections = {
    "processing_image": "Processing image",
    "color_space": "Using color space",
    "image_arm": "Using image ARM",
    # "encoder_gain": "Using encoder gain",
    # "multi_region_arm": "Using multi-region image ARM",
}

quantized_pattern = re.compile(
    r"Final results after quantization: Loss: ([\d.eE+-]+), "
    r"Rate NN: ([\d.eE+-]+), Rate Latent: ([\d.eE+-]+), Rate Img: ([\d.eE+-]+)"
)

kodim_pattern = re.compile(r"kodim(\d{2})")

rows = []

for results_dir_str, experiment_label in experiment_results_dirs:
    results_dir = Path(results_dir_str)

    if not results_dir.exists():
        print(f"[MISSING FOLDER] {results_dir}")
        continue

    log_files = sorted(results_dir.glob("*.log"))
    if not log_files:
        print(f"[NO LOG FILES] {results_dir}")
        continue

    for log_path in log_files:
        text = log_path.read_text(encoding="utf-8", errors="replace")

        missing_sections = [
            name for name, marker in required_text_sections.items() if marker not in text
        ]

        quantized_match = quantized_pattern.search(text)
        if quantized_match is None:
            missing_sections.append("final_quantized_metrics")
        
        kodim_match = kodim_pattern.search(log_path.name)
        if kodim_match is None:
            missing_sections.append("kodim_two_digit_id")

        if missing_sections:
            print(
                f"[SKIP] {log_path.name} | missing: {', '.join(missing_sections)}"
            )
            continue

        loss, rate_nn, rate_latent, rate_img = map(float, quantized_match.groups())
        kodim_id = int(kodim_match.group(1))

        rows.append(
            {
                "experiment": experiment_label,
                "kodim_id": kodim_id,
                "log_file": log_path.name,
                "loss": loss,
                "rate_nn": rate_nn,
                "rate_latent": rate_latent,
                "rate_img": rate_img,
            }
        )

if not rows:
    print("No valid log files were parsed.")
else:
    df = pd.DataFrame(rows).sort_values(["experiment", "kodim_id"]).reset_index(drop=True)

    # Space-separated table output (easy to copy).
    print(df.to_string(index=False, float_format=lambda x: f"{x:.9f}"))


[SKIP] 2026_03_11__02_56_39__coolchic_kodak_kodim11_job_10.log | missing: final_quantized_metrics
[SKIP] 2026_03_11__02_58_39__coolchic_kodak_kodim12_job_11.log | missing: final_quantized_metrics
[SKIP] 2026_03_11__12_08_39__coolchic_kodak_kodim13_job_12.log | missing: final_quantized_metrics
[SKIP] 2026_03_11__20_57_10__coolchic_kodak_kodim19_job_18.log | missing: final_quantized_metrics
[SKIP] 2026_03_11__20_57_16__coolchic_kodak_kodim20_job_19.log | missing: final_quantized_metrics
[SKIP] 2026_03_11__21_28_20__coolchic_kodak_kodim21_job_20.log | missing: final_quantized_metrics
[SKIP] 2026_03_11__21_28_20__coolchic_kodak_kodim22_job_21.log | missing: final_quantized_metrics
[SKIP] 2025_12_04__06_09_26__coolchic_kodak_kodim22.log | missing: final_quantized_metrics
[SKIP] 2025_12_04__06_23_31__coolchic_kodak_kodim23.log | missing: final_quantized_metrics
[SKIP] 2025_12_04__06_29_03__coolchic_kodak_kodim24.log | missing: final_quantized_metrics
                                  experim